# Getting all available Swiss supermarkets using the overpass Web API

**This Jupyter notebook does the following:**

- Queries all supermarkets in Switzerland from OpenStreetMap via the Swiss Overpass Web API (with several attempts if the server is busy)
- Saves the raw data to `supermarkets.json` and converts it to a pandas data frame
- Plots the locations of Migros and Coop supermarkets on an interactive folium map

## Libraries and settings

In [ ]:
# Libraries
import os
import time
import requests
import json
import folium
from pandas import json_normalize

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Show current working directory
print(os.getcwd())

## Overpass turbo query to get all available supermarkets in Switzerland

In [ ]:
# Overpass API URL (Swiss Overpass instance, covers Switzerland only)
url = 'https://overpass.osm.ch/api/interpreter'

# Overpass turbo query (nwr = nodes, ways and relations; 'out center' returns the
# center of supermarkets which are mapped as buildings)
query = f"""
        [out:json][timeout:180];
        area["ISO3166-1"="CH"][admin_level=2];
        nwr["shop"="supermarket"](area);
        out center;"""

# Web API request (several attempts, as the server may be busy)
max_attempts = 5
data = None
for attempt in range(1, max_attempts + 1):
    try:
        r = requests.get(url, params={'data': query}, timeout=300)
        r.raise_for_status()
        data = r.json()['elements']
        print(f'Attempt {attempt}: success')
        break
    except (requests.exceptions.RequestException, ValueError, KeyError) as e:
        print(f'Attempt {attempt} of {max_attempts} failed: {e}')
        if attempt < max_attempts:
            time.sleep(10 * attempt)

if data is None:
    raise RuntimeError('Overpass API request failed after several attempts.')

# Use the center of ways/relations as their location (lat/lon) and drop the node lists
for element in data:
    if 'center' in element:
        element['lat'] = element['center']['lat']
        element['lon'] = element['center']['lon']
        del element['center']
    element.pop('nodes', None)
    element.pop('members', None)

# Save data to file
with open('supermarkets.json', 'w') as json_file:
    json.dump(data, json_file)

# Store data in data frame
df = json_normalize(data)

# Number of rows and columns
print(df.shape)

# First rows
df.head()

## Plot supermarkets on map

In [ ]:
# Subset of supermarkets by brand
locations = df[["lat", "lon", "tags.brand", "tags.shop"]].loc[df["tags.brand"].isin(['Migros', 'Coop'])]
print(locations.head(5))

# Create map
map = folium.Map(location=[locations.lat.mean(), 
                           locations.lon.mean()], 
                 zoom_start=8, 
                 control_scale=True)

# Add maker symbols
for index, location_info in locations.iterrows():
    folium.Marker([location_info["lat"], 
                   location_info["lon"]], 
                  popup=location_info["tags.brand"]).add_to(map)

# Plot map
map

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')